# Model Training

This notebook trains and evaluates multiple machine learning models for phishing URL detection.

## Models to Train:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. Decision Tree
5. Multi-Layer Perceptron (MLP)

In [ ]:
# Import necessary libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

# Import project modules
from src.preprocessing.data_loader import load_dataset, split_data, get_feature_names
from src.preprocessing.preprocessor import PhishingPreprocessor, save_processed_splits
from src.model_training.trainer import run_all_models

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s – %(message)s",
    datefmt="%H:%M:%S",
)

print("Libraries imported successfully.")

## 1. Load Dataset

In [ ]:
# Load the dataset
DATA_PATH = '../data/raw/phishing_dataset.csv'
TARGET_COLUMN = 'CLASS_LABEL'

df = load_dataset(DATA_PATH)
feature_names = get_feature_names(df, TARGET_COLUMN)

print(f"Dataset loaded: {df.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nFeature names: {feature_names[:10]}...")

## 2. Split Data

In [ ]:
# Split into train/validation/test (70/15/15)
X_train_raw, X_val_raw, X_test_raw, \
y_train_raw, y_val_raw, y_test_raw = split_data(df, TARGET_COLUMN)

print(f"Train set: {X_train_raw.shape[0]} samples")
print(f"Validation set: {X_val_raw.shape[0]} samples")
print(f"Test set: {X_test_raw.shape[0]} samples")

## 3. Preprocess Data

In [ ]:
# Initialize preprocessor
preprocessor = PhishingPreprocessor()

# Encode labels
y_train = preprocessor.encode_labels(y_train_raw)
y_val = preprocessor.encode_labels(y_val_raw)
y_test = preprocessor.encode_labels(y_test_raw)

# Scale features (fit on train only)
X_train = preprocessor.fit_transform(X_train_raw, feature_names)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Preprocessed train shape: {X_train.shape}")
print(f"Preprocessed val shape: {X_val.shape}")
print(f"Preprocessed test shape: {X_test.shape}")

## 4. Save Preprocessor

In [ ]:
# Save preprocessor for later use
preprocessor.save('../models_saved')
print("Preprocessor saved to models_saved/preprocessor.pkl")

## 5. Save Processed Splits

In [ ]:
# Save processed splits as CSV files
save_processed_splits(
    (X_train, X_val, X_test, y_train, y_val, y_test),
    feature_names,
    output_dir='../data/processed'
)
print("Processed splits saved to data/processed/")

## 6. Train All Models

In [ ]:
# Train all 5 models with hyperparameter tuning
results = run_all_models(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
)

print("\nModel training complete!")

## 7. Results Summary

In [ ]:
# Display results summary
results_df = pd.DataFrame([
    {
        'Model': r['name'],
        'Test Accuracy': r['accuracy'],
    }
    for r in results
])

results_df = results_df.sort_values('Test Accuracy', ascending=False)
print("\nModel Performance Summary:")
print(results_df.to_string(index=False))

# Find best model
best_model = results_df.iloc[0]
print(f"\nBest Model: {best_model['Model']} with accuracy {best_model['Test Accuracy']:.4f}")

## 8. Model Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot model comparison
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=results_df,
    x='Test Accuracy',
    y='Model',
    palette='viridis',
    ax=ax
)
ax.set_title('Model Comparison - Test Accuracy', fontweight='bold')
ax.set_xlabel('Accuracy')
ax.set_xlim(0.8, 1.0)
ax.spines[['top', 'right']].set_visible(False)

# Add accuracy labels
for i, v in enumerate(results_df['Test Accuracy']):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

## Summary

Model training complete! All models have been saved to `models_saved/` directory:

- logistic_regression.pkl
- random_forest.pkl
- xgboost.pkl
- decision_tree.pkl
- mlp.pkl

Confusion matrices and classification reports have been saved to `reports/figures/`.